# 🛡️ Je construis mon harness

Aujourd'hui, tu passes de l'autre côté : tu vas **construire l'équipement** d'une IA, pièce par pièce.

**Agent = Modèle + Harness.** Le harness a 5 pièces : 🔁 la boucle · 🛠️ les outils · 📖 le contexte · 🚦 les permissions · 🛑 l'arrêt.

Règles du binôme : un au clavier, un qui lit les consignes. **On échange à chaque étape.**
Si une ligne te semble mystérieuse, demande au chat du club : *« Explique-moi ce que fait chaque ligne de ce code : … »*

## Étape 0 — Se brancher au modèle

Exécute la cellule (Maj + Entrée). Elle te demande la clé API donnée par l'animateur. La clé est **secrète** : elle ne s'affiche pas.

*Pas d'internet ? Remplace l'URL par `http://localhost:11434/v1/chat/completions` et le modèle par `qwen3.5:2b` (IA locale).*

In [ ]:
import getpass, json, subprocess, sys
import requests

URL = "https://api.mistral.ai/v1/chat/completions"
MODELE = "mistral-small-latest"
CLE = getpass.getpass("Clé API : ")

def appeler(messages, outils=None):
    """Une ÉTAPE : on envoie du texte au modèle, il renvoie du texte. C'est tout ce qu'il sait faire."""
    corps = {"model": MODELE, "messages": messages}
    if outils:
        corps["tools"] = outils
    r = requests.post(URL, headers={"Authorization": f"Bearer {CLE}"}, json=corps, timeout=120)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]

print("Prêt !")

## Étape 1 — Le modèle seul 🧠

On lui envoie **une** question. Change la question et relance plusieurs fois.

In [ ]:
messages = [{"role": "user", "content": "Explique-moi en 2 phrases ce qu'est un volcan."}]
reponse = appeler(messages)
print(reponse["content"])

✏️ **À toi** : pose-lui une question à laquelle il ne peut **pas** savoir répondre (ce que tu as mangé ce matin, le prénom de ton voisin…). Que fait-il ?

In [ ]:
messages = [{"role": "user", "content": "..."}]
print(appeler(messages)["content"])

## Étape 2 — Le contexte 📖

Le message `system` est lu par le modèle **avant** ta question. C'est toi qui choisis ce qu'il contient.
Même modèle, même question… mais un contexte différent.

In [ ]:
question = {"role": "user", "content": "Combien font 7 fois 8 ?"}

for consigne in ["Tu es un pirate. Tu parles toujours comme un pirate.",
                 "Tu es un prof de maths qui ne donne JAMAIS la réponse, seulement un indice."]:
    messages = [{"role": "system", "content": consigne}, question]
    print("CONSIGNE :", consigne)
    print("RÉPONSE  :", appeler(messages)["content"], "\n")

✏️ **À toi** : invente ta propre consigne (un robot poli, un chat qui miaule, un coach de foot…).

🤔 **Question** : comment Le Chat ou ChatGPT connaissent-ils la date du jour ? (indice : regarde le mot `system`)

In [ ]:
messages = [{"role": "system", "content": "..."},
            {"role": "user", "content": "..."}]
print(appeler(messages)["content"])

## Étape 3 — Le défi du refuge 🐱

Le refuge a un fichier `refuge.csv` avec ses animaux. Question : **combien de chats attendent encore d'être adoptés, et comment s'appelle le plus âgé ?**

Le modèle seul ne peut pas savoir : il n'a jamais vu ce fichier. Vérifie-le :

In [ ]:
QUESTION = "Combien de chats du refuge attendent encore d'être adoptés, et comment s'appelle le plus âgé d'entre eux ?"
print(appeler([{"role": "user", "content": QUESTION}])["content"])

## Étape 4 — Un outil 🛠️

On décrit au modèle un outil : `executer_python`. Le modèle ne connaît **que la description** de l'outil.

Regarde bien : le modèle ne lance rien lui-même. Il écrit une **demande**.

In [ ]:
OUTIL_PYTHON = {
    "type": "function",
    "function": {
        "name": "executer_python",
        "description": "Exécute du code Python et renvoie ce qu'il affiche avec print().",
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string", "description": "Le code Python à exécuter."}},
            "required": ["code"],
        },
    },
}

CONTEXTE = ("Les données du refuge sont dans le fichier 'refuge.csv'. Colonnes : nom, espece, age_ans, poids_kg, "
            "arrivee, adopte (oui/non), caractere. Utilise le module csv. Calcule avec l'outil, ne devine jamais.")

messages = [{"role": "system", "content": CONTEXTE}, {"role": "user", "content": QUESTION}]
reponse = appeler(messages, outils=[OUTIL_PYTHON])

demande = reponse["tool_calls"][0]
print("Le modèle DEMANDE l'outil :", demande["function"]["name"])
print(json.loads(demande["function"]["arguments"])["code"])

🤔 Le code ci-dessus a-t-il été exécuté ? **Non !** C'est seulement une demande. C'est au harness — donc à toi — de décider.

Voici l'outil, côté harness : une fonction qui exécute le code et renvoie ce qu'il affiche.

In [ ]:
def executer_python(code):
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=20)
    return (r.stdout + r.stderr).strip() or "(rien n'a été affiché)"

print(executer_python(json.loads(demande["function"]["arguments"])["code"]))

## Étape 5 — La boucle 🔁

Il faut maintenant **renvoyer le résultat** au modèle, et recommencer tant qu'il demande des outils.
Quand il répond sans demander d'outil, c'est fini : c'est l'**arrêt**.

In [ ]:
def agent(question, etapes_max=8):
    messages = [{"role": "system", "content": CONTEXTE}, {"role": "user", "content": question}]

    for etape in range(1, etapes_max + 1):                        # 🔁 la boucle
        reponse = appeler(messages, outils=[OUTIL_PYTHON])
        if not reponse.get("tool_calls"):                         # 🛑 l'arrêt : il a répondu
            print(f"[étape {etape}] RÉPONSE :", reponse["content"])
            return

        messages.append({"role": "assistant", "content": reponse.get("content") or "",
                         "tool_calls": reponse["tool_calls"]})

        for demande in reponse["tool_calls"]:                     # 🛠️ les outils
            code = json.loads(demande["function"]["arguments"])["code"]
            print(f"[étape {etape}] le modèle demande :\n{code}\n")
            resultat = executer_python(code)
            print(f"[étape {etape}] résultat : {resultat}\n")
            messages.append({"role": "tool", "name": "executer_python",
                             "tool_call_id": demande["id"], "content": resultat})

    print("🛑 Arrêt : trop d'étapes.")

agent(QUESTION)

✅ La bonne réponse est **7 chats, le plus âgé est Noisette (11 ans)**. Attention au piège : Gribouille (13 ans) est déjà adopté !
Ton agent a-t-il trouvé ? Sinon, comment pourrais-tu l'aider ? (indice : le contexte)

✏️ **À toi** : pose une autre question sur le refuge à ton agent (le chien le plus lourd ? combien de lapins ?).

In [ ]:
agent("...")

## Étape 6 — Les permissions 🚦 et l'arrêt 🛑

Un agent qui exécute n'importe quel code sans demander, c'est dangereux : il pourrait effacer des fichiers !

✏️ **À toi** : modifie la fonction `agent` (recopie-la ci-dessous) pour que :
1. avant d'exécuter un code, le harness demande `input("Autoriser ? (o/n) ")`, et si la réponse n'est pas `o`,
   le résultat renvoyé au modèle est `"Refusé par l'humain."` ;
2. l'agent s'arrête après **5** étapes au lieu de 8.

Puis teste : refuse la première demande. Que fait le modèle ?

In [ ]:
# Recopie et modifie la fonction agent ici

## 🏆 Défi bonus — Le harness de mon projet

Écris le **contexte** idéal pour une IA qui aiderait ton équipe sur votre projet : qui vous êtes, ce que vous construisez,
et vos règles (« ne donne jamais le code entier », « explique comme à un enfant de 11 ans »…).
Teste-le avec une vraie question sur votre projet. Si ça marche bien, recopie-le dans votre feuille de route !

In [ ]:
MON_CONTEXTE = """
...
"""
messages = [{"role": "system", "content": MON_CONTEXTE},
            {"role": "user", "content": "..."}]
print(appeler(messages)["content"])

## 📓 Pour ton carnet

- Quelle pièce du harness as-tu préférée construire ? Pourquoi ?
- Explique avec tes mots : *« Le modèle ne lance jamais rien lui-même… »*